In [38]:
import polars as pl
import math
from pathlib import Path
from datetime import date
from datetime import timedelta

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(250)

polars.config.Config

In [ ]:
GTFS = Path("raw/processed_gtfs")
RT = Path("raw")
 
# ---------------------------------------------------
# Read files
# ---------------------------------------------------

routes = pl.read_parquet(GTFS / "routes.parquet")
trips = pl.read_parquet(GTFS / "trips.parquet")
stops = pl.read_parquet(GTFS / "stops.parquet")
stop_times = pl.read_parquet(GTFS / "stop_times.parquet")

# trip_updates = pl.read_parquet(RT / "trip_updates/2026-03-12.parquet")
# vehicle_positions = pl.read_parquet(RT / "vehicle_positions/2026-03-12.parquet")
 
print("Loaded")

Loaded


In [ ]:
# %%
# =====================================================================
# REPLACES your current single-date trip_updates/vehicle_positions read
# cell. Auto-detects every date available in BOTH folders within your
# target range, so it stays correct as more days get fetched later -
# no need to hand-maintain a date list.
# =====================================================================

RT = Path("raw")
DATE_START = date(2026, 3, 3)
DATE_END = date(2026, 7, 3)


def available_dates(dir_path: Path) -> set[str]:
    return {p.stem for p in dir_path.glob("*.parquet")}


tu_dates = available_dates(RT / "trip_updates")
vp_dates = available_dates(RT / "vehicle_positions")
both_dates = tu_dates & vp_dates

usable_dates = sorted(
    d for d in both_dates
    if DATE_START.isoformat() <= d <= DATE_END.isoformat()
)
print(f"trip_updates dates found: {len(tu_dates)} | vehicle_positions dates found: {len(vp_dates)}")
print(f"usable dates (present in both, within range): {len(usable_dates)}")

missing = [
    (DATE_START + __import__("datetime").timedelta(days=i)).isoformat()
    for i in range((DATE_END - DATE_START).days + 1)
]
missing = sorted(set(missing) - set(usable_dates))
if missing:
    print(f"NOTE: {len(missing)} days in range have no usable RT data yet "
          f"(missing from one or both feeds) - proceeding with what's available.")
    print("first few missing:", missing[:10])

trip_updates = pl.concat([
    pl.read_parquet(RT / "trip_updates" / f"{d}.parquet") for d in usable_dates
])
vehicle_positions = pl.concat([
    pl.read_parquet(RT / "vehicle_positions" / f"{d}.parquet") for d in usable_dates
])

print(f"trip_updates: {trip_updates.shape} | vehicle_positions: {vehicle_positions.shape}")

In [41]:
def gtfs_to_seconds(col):
    p = pl.col(col).str.split(":")
    return (
        p.list.get(0).cast(pl.Int32) * 3600
        + p.list.get(1).cast(pl.Int32) * 60
        + p.list.get(2).cast(pl.Int32)
    )
 
stop_times = stop_times.with_columns(
    pl.col("stop_sequence").cast(pl.UInt32),
    pl.col("stop_id").cast(pl.Utf8),
    gtfs_to_seconds("arrival_time").alias("scheduled_arrival"),
    gtfs_to_seconds("departure_time").alias("scheduled_departure"),
)
 
stops = stops.with_columns([
    pl.col("stop_id").cast(pl.Utf8),

    # pl.col("stop_lat")
    #   .str.strip_chars()
    #   .cast(pl.Float64),

    # pl.col("stop_lon")
    #   .str.strip_chars()
    #   .cast(pl.Float64),
])

In [42]:

trip_updates = (
    trip_updates
    .sort("feed_timestamp")
    .group_by(["trip_id", "start_date", "stop_sequence"])
    .last()
)

In [43]:
if "schedule_relationship" in trip_updates.columns:
    trip_updates = trip_updates.filter(pl.col("schedule_relationship") == 0)
 

In [44]:
trip_updates = trip_updates.with_columns(
    pl.from_epoch("arrival_time", time_unit="s").alias("event_time")
)
 
vehicle_positions = vehicle_positions.with_columns(
    pl.from_epoch("timestamp", time_unit="s").alias("vehicle_time")
)

In [45]:
TRIP_ID_PATTERN = r'^[A-Z]{2}_([A-Z0-9]+)-(\w+?)-(\d+)_([A-Z0-9+]+)_(\d+)$'
 
def parse_trip_id(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col("trip_id").str.extract(TRIP_ID_PATTERN, 2).alias("_service_day"),
        pl.col("trip_id").str.extract(TRIP_ID_PATTERN, 3).alias("_origin_secs"),
        pl.col("trip_id").str.extract(TRIP_ID_PATTERN, 5).alias("_trip_num"),
    ])
 
trips_parsed = parse_trip_id(trips)
tu_parsed = parse_trip_id(trip_updates)
 
# 1) direct trip_id match
direct = tu_parsed.join(
    trips_parsed.select(["trip_id", "route_id", "direction_id", "shape_id", "service_id"]),
    on="trip_id", how="inner", suffix="_static",
)

In [46]:
unmatched = tu_parsed.join(direct.select("trip_id").unique(), on="trip_id", how="anti")
fallback = unmatched.join(
    trips_parsed.select([
        "trip_id", "route_id", "direction_id", "shape_id", "service_id",
        "_service_day", "_origin_secs", "_trip_num",
    ]).rename({"trip_id": "_static_trip_id"}),
    on=["route_id", "_service_day", "_origin_secs", "_trip_num"],
    how="inner",
    suffix="_static",
).with_columns(
    pl.col("_static_trip_id").alias("trip_id")   # use the STATIC trip_id downstream
).drop("_static_trip_id")
 
n_direct, n_fallback, n_total = direct.height, fallback.height, tu_parsed.height
print(f"[join] direct match: {n_direct:,} | fallback match: {n_fallback:,} | "
      f"total: {(n_direct + n_fallback) / n_total:.1%} of {n_total:,} rows")
 
direct = direct.drop(["_service_day", "_origin_secs", "_trip_num"])
fallback = fallback.drop(["_service_day", "_origin_secs", "_trip_num"])

[join] direct match: 71,266 | fallback match: 0 | total: 100.0% of 71,266 rows


In [47]:
# %%
# =====================================================================
# CALENDAR VALIDITY CHECK - insert right after your existing trip_id
# join (direct + fallback), before the stop_times join. Verifies each
# matched (service_id, start_date) pair is actually a real service day,
# not just a string match - important now that trips/calendar/
# calendar_dates span MULTIPLE rating periods concatenated together.
# =====================================================================
 
calendar = pl.read_parquet("raw/processed_gtfs/calendar.parquet")
calendar_dates = pl.read_parquet("raw/processed_gtfs/calendar_dates.parquet")
 
 
def resolve_service_dates(calendar_df: pl.DataFrame, calendar_dates_df: pl.DataFrame) -> pl.DataFrame:
    from datetime import date as _date, timedelta as _timedelta
    weekday_cols = ["monday", "tuesday", "wednesday", "thursday", "friday", "saturday", "sunday"]
    rows = []
    for row in calendar_df.iter_rows(named=True):
        start = _date(int(str(row["start_date"])[:4]), int(str(row["start_date"])[4:6]), int(str(row["start_date"])[6:8]))
        end = _date(int(str(row["end_date"])[:4]), int(str(row["end_date"])[4:6]), int(str(row["end_date"])[6:8]))
        d = start
        while d <= end:
            if row[weekday_cols[d.weekday()]] == 1:
                rows.append({"service_id": row["service_id"], "date": int(d.strftime("%Y%m%d"))})
            d += _timedelta(days=1)
    base = pl.DataFrame(rows)
    additions = calendar_dates_df.filter(pl.col("exception_type") == 1).select(["service_id", "date"])
    removals = calendar_dates_df.filter(pl.col("exception_type") == 2).select(["service_id", "date"])
    resolved = pl.concat([base, additions]).unique()
    resolved = resolved.join(removals, on=["service_id", "date"], how="anti")
    return resolved
 
 
service_dates = resolve_service_dates(calendar, calendar_dates)
print(f"resolved {service_dates.height:,} valid (service_id, date) pairs spanning "
      f"{service_dates['date'].min()} - {service_dates['date'].max()}")

resolved 1,190 valid (service_id, date) pairs spanning 20260103 - 20260905


In [48]:
direct = (
    direct.drop(["route_id", "direction_id"])
          .rename({"route_id_static": "route_id", "direction_id_static": "direction_id"})
)
fallback = fallback.drop("direction_id").rename({"direction_id_static": "direction_id"})
 
matched = pl.concat([direct, fallback.select(direct.columns)])

In [49]:
data = (
    matched
    .join(
        stop_times.select([
            "trip_id", "stop_sequence", "stop_id",
            "scheduled_arrival", "scheduled_departure",
        ]),
        on=["trip_id", "stop_sequence"],
        how="inner",   # inner now: every row here already has a resolved static trip_id,
                        # so a missing stop_times row means bad data, not an expected gap
    )
    .join(
        stops.select(["stop_id", "stop_lat", "stop_lon"]),
        on="stop_id",
        how="left",
    )
)
 
print(data.shape)

(71255, 22)


In [50]:
data = data.with_columns(pl.col("start_date").cast(pl.Int64).alias("_start_date_int"))
 
invalid = data.join(
    service_dates.rename({"date": "_start_date_int"}),
    on=["service_id", "_start_date_int"],
    how="anti",
)
print(f"rows with a matched trip_id but an INVALID (service_id, date) pair: {invalid.height:,} "
      f"({invalid.height / max(data.height, 1):.2%} of {data.height:,})")
if invalid.height > 0:
    print("sample of invalid rows (worth checking which rating period they came from):")
    print(invalid.select(["trip_id", "route_id", "service_id", "start_date"]).head(10))
 
data = data.join(
    service_dates.rename({"date": "_start_date_int"}),
    on=["service_id", "_start_date_int"],
    how="semi",
).drop("_start_date_int")
 
print(f"data after calendar validity filter: {data.shape}")

rows with a matched trip_id but an INVALID (service_id, date) pair: 0 (0.00% of 71,255)
data after calendar validity filter: (71255, 22)


In [51]:
def haversine_expr(lat1, lon1, lat2, lon2) -> pl.Expr:
    R = 6371000.0
    lat1r, lat2r = lat1.radians(), lat2.radians()
    dlat = (lat2 - lat1).radians()
    dlon = (lon2 - lon1).radians()
    a = (dlat / 2).sin() ** 2 + lat1r.cos() * lat2r.cos() * (dlon / 2).sin() ** 2
    return 2 * R * a.sqrt().arcsin()
 
vehicle_positions = vehicle_positions.sort(["vehicle_id", "vehicle_time"])
vehicle_positions = vehicle_positions.with_columns([
    pl.col("latitude").shift(1).over("vehicle_id").alias("_prev_lat"),
    pl.col("longitude").shift(1).over("vehicle_id").alias("_prev_lon"),
    pl.col("vehicle_time").shift(1).over("vehicle_id").alias("_prev_time"),
])
vehicle_positions = vehicle_positions.with_columns(
    haversine_expr(pl.col("_prev_lat"), pl.col("_prev_lon"), pl.col("latitude"), pl.col("longitude")).alias("_dist_m")
)
vehicle_positions = vehicle_positions.with_columns(
    (
        pl.col("_dist_m")
        / (pl.col("vehicle_time") - pl.col("_prev_time")).dt.total_seconds().clip(lower_bound=1)
    ).alias("speed_mps")
)

In [52]:
vp = vehicle_positions.select([
    "vehicle_id", "vehicle_time", "latitude", "longitude", "bearing", "speed_mps",
])
 
data = data.sort(["vehicle_id", "event_time"])
vp = vp.sort(["vehicle_id", "vehicle_time"])
 
data = data.join_asof(
    vp,
    left_on="event_time",
    right_on="vehicle_time",
    by="vehicle_id",
    strategy="backward",
    tolerance=timedelta(minutes=2),
)

C:\Users\ishan\AppData\Local\Temp\ipykernel_7684\3507643840.py:8: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  data = data.join_asof(


In [53]:
data = (
    data
    .sort(["trip_id", "stop_sequence"])
    .with_columns(
        pl.col("arrival_time").shift(-1).over("trip_id").alias("next_arrival_time")
    )
    .with_columns(
        (pl.col("next_arrival_time") - pl.col("arrival_time")).alias("travel_time")
    )
)
 
data = data.filter(
    (pl.col("travel_time") > 0) & (pl.col("travel_time") < 1800)   # 30 min cap
)

In [54]:
# data = data.with_columns([
#     pl.col("event_time").dt.hour().alias("hour"),
#     pl.col("event_time").dt.weekday().alias("weekday"),
#     pl.col("event_time").dt.month().alias("month"),
# ])
 
# data = data.with_columns([
#     (
#         (pl.col("hour").is_between(7, 9)) |
#         (pl.col("hour").is_between(16, 18))
#     ).cast(pl.Int8).alias("is_peak")
# ])

data = data.with_columns(
    pl.col("event_time")
      .dt.replace_time_zone("UTC")
      .dt.convert_time_zone("America/New_York")
      .alias("event_time_local")
)
 
data = data.with_columns([
    pl.col("event_time_local").dt.hour().alias("hour"),
    pl.col("event_time_local").dt.weekday().alias("weekday"),
    pl.col("event_time_local").dt.month().alias("month"),
])
 
data = data.with_columns([
    (
        (pl.col("hour").is_between(7, 9)) |
        (pl.col("hour").is_between(16, 18))
    ).cast(pl.Int8).alias("is_peak")
])

In [55]:
data = data.with_columns([
    (
        pl.col("scheduled_arrival")
        - pl.col("scheduled_departure").shift(1).over("trip_id")
    ).alias("scheduled_segment_time"),
 
    (
        pl.col("stop_sequence") / pl.col("stop_sequence").max().over("trip_id")
    ).alias("trip_progress"),
])

In [56]:
segment_network = pl.read_parquet("processed/segment_network.parquet")
 
# dtype fix: segment_network's stop_id/next_stop_id are Int64 (straight
# from stop_times.txt), but `data`'s stop_id was cast to Utf8 earlier for
# the RT join - cast both sides to match before joining.
segment_network = segment_network.with_columns([
    pl.col("stop_id").cast(pl.Utf8),
    pl.col("next_stop_id").cast(pl.Utf8),
])

In [57]:
data = (
    data
    .sort(["trip_id", "stop_sequence"])
    .with_columns(
        pl.col("stop_id").shift(-1).over("trip_id").alias("next_stop_id")
    )
)
 
# %%
n_before = data.height
 
data = data.join(
    segment_network.select([
        "shape_id", "stop_id", "next_stop_id",
        "segment_length", "scheduled_travel_time",
    ]).rename({"scheduled_travel_time": "segment_scheduled_travel_time"}),
    on=["shape_id", "stop_id", "next_stop_id"],
    how="left",
)
 
print(f"rows before: {n_before:,} | after segment join: {data.height:,}")
print("null segment_length rows:", data.filter(pl.col("segment_length").is_null()).height)
 

rows before: 62,783 | after segment join: 62,783
null segment_length rows: 3699


In [58]:
data = data.with_columns(
    (pl.col("segment_length") / pl.col("segment_scheduled_travel_time").clip(lower_bound=1))
    .alias("scheduled_segment_speed_mps")
)

In [59]:
data = data.sort(["trip_id", "stop_sequence"])
check = data.select([
    "trip_id", "stop_sequence", "scheduled_segment_time", "segment_scheduled_travel_time",
]).with_columns(
    # bring segment_scheduled_travel_time from row k up to align with
    # scheduled_segment_time at row k+1 (both now describe segment k->k+1)
    pl.col("segment_scheduled_travel_time").shift(1).over("trip_id").alias("segment_scheduled_travel_time_aligned")
).drop_nulls(subset=["scheduled_segment_time", "segment_scheduled_travel_time_aligned"])
 
diff = (check["scheduled_segment_time"] - check["segment_scheduled_travel_time_aligned"]).abs()
print("median abs diff (aligned):", diff.median())
print("rows with diff > 30s (aligned):", (diff > 30).sum(), "/", check.height)
 
# expected: last stop of each trip has null segment_length/next_stop_id -
# should be roughly one per trip, not a bug
n_null_segment = data.filter(pl.col("segment_length").is_null()).height
n_trips = data["trip_id"].n_unique()
print(f"null segment_length rows: {n_null_segment:,} vs trip count: {n_trips:,} (should be close)")
 

median abs diff (aligned): 20.0
rows with diff > 30s (aligned): 20552 / 59084
null segment_length rows: 3,699 vs trip count: 1,219 (should be close)


In [60]:

import requests
from datetime import datetime
 
STATION_LAT, STATION_LON = 40.7829, -73.9654  # Central Park
 
dates_needed = data["start_date"].unique().sort().to_list()
start_date = str(dates_needed[0])
end_date = str(dates_needed[-1])
start_fmt = f"{start_date[:4]}-{start_date[4:6]}-{start_date[6:8]}"
end_fmt = f"{end_date[:4]}-{end_date[4:6]}-{end_date[6:8]}"
 
print(f"Fetching weather for {start_fmt} to {end_fmt}")
 
resp = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": STATION_LAT,
        "longitude": STATION_LON,
        "start_date": start_fmt,
        "end_date": end_fmt,
        "hourly": "temperature_2m,precipitation,rain,snowfall,"
                  "windspeed_10m,weathercode",
        "timezone": "America/New_York",
    },
    timeout=30,
)
resp.raise_for_status()
weather_json = resp.json()["hourly"]
 
weather = pl.DataFrame({
    "weather_time_str": weather_json["time"],
    "temperature_c": weather_json["temperature_2m"],
    "precipitation_mm": weather_json["precipitation"],
    "rain_mm": weather_json["rain"],
    "snowfall_cm": weather_json["snowfall"],
    "windspeed_kmh": weather_json["windspeed_10m"],
    "weathercode": weather_json["weathercode"],
})
 
weather = weather.with_columns(
    pl.col("weather_time_str").str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M").alias("weather_time")
).with_columns([
    pl.col("weather_time").dt.strftime("%Y%m%d").alias("start_date"),  # keep as str - matches data's dtype
    pl.col("weather_time").dt.hour().alias("hour"),
])
 
print(weather.shape)
print(weather.head())

Fetching weather for 2026-03-11 to 2026-03-12
(48, 10)
shape: (5, 10)
┌──────────────────┬───────────────┬──────────────────┬─────────┬─────────────┬───────────────┬─────────────┬─────────────────────┬────────────┬──────┐
│ weather_time_str ┆ temperature_c ┆ precipitation_mm ┆ rain_mm ┆ snowfall_cm ┆ windspeed_kmh ┆ weathercode ┆ weather_time        ┆ start_date ┆ hour │
│ ---              ┆ ---           ┆ ---              ┆ ---     ┆ ---         ┆ ---           ┆ ---         ┆ ---                 ┆ ---        ┆ ---  │
│ str              ┆ f64           ┆ f64              ┆ f64     ┆ f64         ┆ f64           ┆ i64         ┆ datetime[μs]        ┆ str        ┆ i8   │
╞══════════════════╪═══════════════╪══════════════════╪═════════╪═════════════╪═══════════════╪═════════════╪═════════════════════╪════════════╪══════╡
│ 2026-03-11T00:00 ┆ 16.1          ┆ 0.0              ┆ 0.0     ┆ 0.0         ┆ 8.3           ┆ 0           ┆ 2026-03-11 00:00:00 ┆ 20260311   ┆ 0    │
│ 2026-03-11T01:00

In [61]:
# %%
# ---------------------------------------------------------------------
# Join weather onto data by (start_date, hour) - both now NYC-local
# ---------------------------------------------------------------------
 
n_before = data.height
 
data = data.join(
    weather.select([
        "start_date", "hour", "temperature_c", "precipitation_mm",
        "rain_mm", "snowfall_cm", "windspeed_kmh", "weathercode",
    ]),
    on=["start_date", "hour"],
    how="left",
)
 
print(f"rows before: {n_before:,} | after weather join: {data.height:,}")
print("null weather rows:", data.filter(pl.col("temperature_c").is_null()).height)

rows before: 62,783 | after weather join: 62,783
null weather rows: 0


In [62]:
data = data.with_columns([
    (pl.col("precipitation_mm") > 0.1).cast(pl.Int8).alias("is_raining"),
    (pl.col("snowfall_cm") > 0.0).cast(pl.Int8).alias("is_snowing"),
    # WMO weather codes 45 (fog) / 48 (depositing rime fog) as a
    # low-visibility proxy - the archive API doesn't expose visibility
    # directly (ERA5 reanalysis has no such variable; that's a
    # forecast-API-only field), so this is the closest available signal.
    pl.col("weathercode").is_in([45, 48]).cast(pl.Int8).alias("is_fog"),
])

In [63]:
# 1. UPSTREAM DELAY
# ---------------------------------------------------------------------
# scheduled_arrival is seconds-since-midnight (local, naive), while
# arrival_time (RT) is an absolute UTC epoch - not directly comparable
# yet. Convert scheduled_arrival to an absolute epoch first, anchored on
# start_date in America/New_York (same pattern used for the weather join).
 
data = data.with_columns(
    pl.col("start_date").str.strptime(pl.Date, "%Y%m%d").alias("service_date")
)
 
data = data.with_columns(
    (
        pl.col("service_date").cast(pl.Datetime)
          .dt.replace_time_zone("America/New_York")
        + pl.duration(seconds=pl.col("scheduled_arrival"))
    ).dt.epoch(time_unit="s").alias("scheduled_arrival_epoch")
)
 
data = data.with_columns(
    (pl.col("arrival_time") - pl.col("scheduled_arrival_epoch")).alias("delay_seconds")
)

In [64]:
# ---------------------------------------------------------------------
# Load + reshape ridership CSV - route + hour granularity
# ---------------------------------------------------------------------
ridership_raw = pl.read_csv(
    "MTA_Bus_Hourly_Ridership_Mar_July.csv",
    schema_overrides={"ridership": pl.Utf8, "transfers": pl.Utf8},
)

# ridership/transfers arrive as strings with thousands separators
# (e.g. "1,131") - strip commas before casting to Int64
ridership_raw = ridership_raw.with_columns(
    pl.col("transit_timestamp")
      .str.strptime(pl.Datetime, "%m/%d/%Y %I:%M:%S %p")
      .alias("_ts"),
    pl.col("ridership").str.replace_all(",", "").cast(pl.Int64),
    pl.col("transfers").str.replace_all(",", "").cast(pl.Int64),
).with_columns([
    pl.col("_ts").dt.date().alias("service_date"),
    pl.col("_ts").dt.hour().alias("hour"),
])

# collapse both fare_class_category and payment_method - one total per
# (route, date, hour)
ridership_hourly = (
    ridership_raw
    .group_by(["bus_route", "service_date", "hour"])
    .agg([
        pl.col("ridership").sum(),
        pl.col("transfers").sum(),
    ])
)

print(ridership_hourly.shape)
print(ridership_hourly.head(10))


(16320, 5)
shape: (10, 5)
┌───────────┬──────────────┬──────┬───────────┬───────────┐
│ bus_route ┆ service_date ┆ hour ┆ ridership ┆ transfers │
│ ---       ┆ ---          ┆ ---  ┆ ---       ┆ ---       │
│ str       ┆ date         ┆ i8   ┆ i64       ┆ i64       │
╞═══════════╪══════════════╪══════╪═══════════╪═══════════╡
│ M4        ┆ 2026-04-09   ┆ 23   ┆ 106       ┆ 13        │
│ M101      ┆ 2026-04-06   ┆ 1    ┆ 21        ┆ 3         │
│ M1        ┆ 2026-03-02   ┆ 21   ┆ 119       ┆ 17        │
│ M15       ┆ 2026-04-13   ┆ 1    ┆ 31        ┆ 9         │
│ M2        ┆ 2026-04-20   ┆ 10   ┆ 336       ┆ 86        │
│ M1        ┆ 2026-04-28   ┆ 23   ┆ 29        ┆ 4         │
│ M101      ┆ 2026-05-13   ┆ 9    ┆ 950       ┆ 321       │
│ M2        ┆ 2026-03-18   ┆ 1    ┆ 7         ┆ 1         │
│ M1        ┆ 2026-03-28   ┆ 18   ┆ 364       ┆ 51        │
│ M1        ┆ 2026-03-21   ┆ 14   ┆ 617       ┆ 146       │
└───────────┴──────────────┴──────┴───────────┴───────────┘


In [65]:
# ---------------------------------------------------------------------
# Join onto `data` by (route_id, service_date, hour).
# NOTE: this ridership file only covers 5 routes (M1, M2, M4, M15, M101).
# route_id naming is verified before joining - if your GTFS route_id has
# an agency prefix (e.g. "MTA NYCT_M1") rather than the bare "M1" used in
# the ridership file, normalize one side before joining.
# ---------------------------------------------------------------------
sample_route_ids = data.select("route_id").unique().sort("route_id").head(10)
print("sample route_id values in `data`:", sample_route_ids["route_id"].to_list())
print("route values in ridership file:", ridership_hourly["bus_route"].unique().sort().to_list())

# If the printed lists above don't share the same format (e.g. one has a
# prefix like "MTA NYCT_"), normalize here before the join, e.g.:
# data = data.with_columns(pl.col("route_id").str.extract(r"([A-Z]+\d+)$", 1).alias("route_id"))

n_before = data.height

data = data.join(
    ridership_hourly,
    left_on=["route_id", "service_date", "hour"],
    right_on=["bus_route", "service_date", "hour"],
    how="left",
).with_columns([
    pl.col("ridership").fill_null(0),
    pl.col("transfers").fill_null(0),
])

print(f"rows before: {n_before:,} | after ridership join: {data.height:,}")
print("routes with zero-filled ridership (not in ridership file, or no match):",
      data.filter(pl.col("ridership") == 0).select("route_id").unique().height)


sample route_id values in `data`: ['M1', 'M101', 'M15', 'M2', 'M4']
route values in ridership file: ['M1', 'M101', 'M15', 'M2', 'M4']
rows before: 62,783 | after ridership join: 62,783
routes with zero-filled ridership (not in ridership file, or no match): 0


In [66]:
print(data.select(
    pl.col("delay_seconds").min().alias("min"),
    pl.col("delay_seconds").max().alias("max"),
    pl.col("delay_seconds").median().alias("median"),
))

shape: (1, 3)
┌───────┬──────┬────────┐
│ min   ┆ max  ┆ median │
│ ---   ┆ ---  ┆ ---    │
│ i64   ┆ i64  ┆ f64    │
╞═══════╪══════╪════════╡
│ -1176 ┆ 5533 ┆ 132.0  │
└───────┴──────┴────────┘


In [67]:
data = data.sort(["trip_id", "stop_sequence"])
data = data.with_columns(
    pl.col("delay_seconds").shift(1).over("trip_id").alias("upstream_delay_seconds")
)


In [68]:
# %%
# ---------------------------------------------------------------------
# 3. HEADWAY TO PRECEDING BUS (same route + direction + stop)
# ---------------------------------------------------------------------
# For each (route_id, direction_id, stop_id), sort arrivals by time and
# take the gap to the PREVIOUS bus that hit the same stop - this is a
# same-route self-join across TRIPS, not within a single trip.
 
headway_base = (
    data.select(["route_id", "direction_id", "stop_id", "trip_id", "arrival_time"])
    .unique()
    .sort(["route_id", "direction_id", "stop_id", "arrival_time"])
)
 
headway_base = headway_base.with_columns(
    pl.col("arrival_time")
      .shift(1)
      .over(["route_id", "direction_id", "stop_id"])
      .alias("_prev_bus_arrival")
)
headway_base = headway_base.with_columns(
    (pl.col("arrival_time") - pl.col("_prev_bus_arrival")).alias("headway_seconds")
)

In [69]:
data = data.join(
    headway_base.select(["route_id", "direction_id", "stop_id", "trip_id", "arrival_time", "headway_seconds"]),
    on=["route_id", "direction_id", "stop_id", "trip_id", "arrival_time"],
    how="left",
)

In [70]:
print(data.select(
    pl.col("headway_seconds").min().alias("min"),
    pl.col("headway_seconds").max().alias("max"),
    pl.col("headway_seconds").median().alias("median"),
))
print("null headway rows (first bus of the day at that stop):",
      data.filter(pl.col("headway_seconds").is_null()).height)
 

shape: (1, 3)
┌─────┬───────┬────────┐
│ min ┆ max   ┆ median │
│ --- ┆ ---   ┆ ---    │
│ i64 ┆ i64   ┆ f64    │
╞═════╪═══════╪════════╡
│ 0   ┆ 58219 ┆ 627.0  │
└─────┴───────┴────────┘
null headway rows (first bus of the day at that stop): 690


In [71]:
calendar_df = pl.read_parquet("calendar_dataset.parquet")
calendar_df = calendar_df.with_columns(
    pl.col("service_date").str.strptime(pl.Date, "%Y-%m-%d")
)

calendar_df = calendar_df.select([
    "service_date",
    "is_weekend",
    "is_federal_holiday",
    "is_school_day",
    "major_event_count",
]).with_columns(
    (pl.col("major_event_count") > 0).alias("has_major_event")
)
 
n_before = data.height
 
data = data.join(calendar_df, on="service_date", how="left")
 
print(f"rows before: {n_before:,} | after calendar join: {data.height:,}")
print("null calendar rows (service_date outside 2026-03-01 to 2026-07-17):",
      data.filter(pl.col("is_federal_holiday").is_null()).height)

rows before: 62,783 | after calendar join: 62,783
null calendar rows (service_date outside 2026-03-01 to 2026-07-17): 0


In [72]:
data

trip_id,start_date,stop_sequence,feed_timestamp,fetch_timestamp,start_time,schedule_relationship,vehicle_id,trip_timestamp,stop_id,arrival_time,departure_time,event_time,route_id,direction_id,shape_id,service_id,stop_id_right,scheduled_arrival,scheduled_departure,stop_lat,stop_lon,vehicle_time,latitude,longitude,bearing,speed_mps,next_arrival_time,travel_time,event_time_local,hour,weekday,month,is_peak,scheduled_segment_time,trip_progress,next_stop_id,segment_length,segment_scheduled_travel_time,scheduled_segment_speed_mps,temperature_c,precipitation_mm,rain_mm,snowfall_cm,windspeed_kmh,weathercode,is_raining,is_snowing,is_fog,service_date,scheduled_arrival_epoch,delay_seconds,ridership,transfers,upstream_delay_seconds,headway_seconds,is_weekend,is_federal_holiday,is_school_day,major_event_count,has_major_event
str,str,u32,u64,"datetime[μs, UTC]",str,i32,str,u64,str,i64,i64,datetime[μs],str,i64,str,str,str,i32,i32,f64,f64,datetime[μs],f32,f32,f32,f64,i64,i64,"datetime[μs, America/New_York]",i8,i8,i8,i8,i32,f64,str,f64,i64,f64,f64,f64,f64,f64,f64,i64,i8,i8,i8,date,i64,i64,i64,i64,i64,i64,bool,bool,bool,u32,bool
"""MV_A6-Weekday-SDon-004000_M2_2…","""20260312""",2,1773290931,2026-03-12 04:49:03.097898 UTC,"""""",0,"""MTA NYCT_9437""",1773290924,"""400242""",1773290942,1773290942,2026-03-12 04:49:02,"""M2""",0,"""M020100""","""MV_A6-Weekday-SDon""","""400242""",2436,2436,40.823169,-73.937752,2026-03-12 04:48:44,40.823627,-73.937523,54.782406,null,1773290990,48,2026-03-12 00:49:02 EDT,0,4,3,0,null,0.181818,"""404129""",139.205048,28,4.971609,17.3,2.2,2.2,0.0,13.9,61,1,0,0,2026-03-12,1773290436,506,60,3,null,66,false,false,true,0,false
"""MV_A6-Weekday-SDon-004000_M2_2…","""20260312""",3,1773290976,2026-03-12 04:50:02.848928 UTC,"""""",0,"""MTA NYCT_9437""",1773290954,"""404129""",1773290990,1773290990,2026-03-12 04:49:50,"""M2""",0,"""M020100""","""MV_A6-Weekday-SDon""","""404129""",2467,2467,40.82426,-73.936943,2026-03-12 04:49:14,40.824047,-73.93721,53.130104,1.785702,1773291003,13,2026-03-12 00:49:50 EDT,0,4,3,0,31,0.272727,"""403338""",171.64037,34,5.048246,17.3,2.2,2.2,0.0,13.9,61,1,0,0,2026-03-12,1773290467,523,60,3,506,97,false,false,true,0,false
"""MV_A6-Weekday-SDon-004000_M2_2…","""20260312""",4,1773291006,2026-03-12 04:50:32.850213 UTC,"""""",0,"""MTA NYCT_9437""",1773290984,"""403338""",1773291003,1773291003,2026-03-12 04:50:03,"""M2""",0,"""M020100""","""MV_A6-Weekday-SDon""","""403338""",2504,2504,40.825612,-73.935955,2026-03-12 04:49:14,40.824047,-73.93721,53.130104,1.785702,1773291046,43,2026-03-12 00:50:03 EDT,0,4,3,0,37,0.363636,"""404837""",118.236767,23,5.140729,17.3,2.2,2.2,0.0,13.9,61,1,0,0,2026-03-12,1773290504,499,60,3,523,81,false,false,true,0,false
"""MV_A6-Weekday-SDon-004000_M2_2…","""20260312""",5,1773291051,2026-03-12 04:51:02.849028 UTC,"""""",0,"""MTA NYCT_9437""",1773291044,"""404837""",1773291046,1773291046,2026-03-12 04:50:46,"""M2""",0,"""M020100""","""MV_A6-Weekday-SDon""","""404837""",2530,2530,40.826531,-73.935264,2026-03-12 04:50:44,40.827339,-73.935211,97.594643,79.063506,1773291217,171,2026-03-12 00:50:46 EDT,0,4,3,0,26,0.454545,"""400248""",779.224571,155,5.027255,17.3,2.2,2.2,0.0,13.9,61,1,0,0,2026-03-12,1773290530,516,60,3,499,99,false,false,true,0,false
"""MV_A6-Weekday-SDon-004000_M2_2…","""20260312""",6,1773291198,2026-03-12 04:53:32.849490 UTC,"""""",0,"""MTA NYCT_9437""",1773291194,"""400248""",1773291217,1773291217,2026-03-12 04:53:37,"""M2""",0,"""M020100""","""MV_A6-Weekday-SDon""","""400248""",2702,2702,40.830904,-73.940279,2026-03-12 04:53:14,40.830944,-73.940338,93.63121,63.665023,1773291252,35,2026-03-12 00:53:37 EDT,0,4,3,0,172,0.545455,"""400249""",243.057468,48,5.063697,17.3,2.2,2.2,0.0,13.9,61,1,0,0,2026-03-12,1773290702,515,60,3,516,87,false,false,true,0,false
"""MV_A6-Weekday-SDon-004000_M2_2…","""20260312""",7,1773291231,2026-03-12 04:54:02.849386 UTC,"""""",0,"""MTA NYCT_9437""",1773291224,"""400249""",1773291252,1773291252,2026-03-12 04:54:12,"""M2""",0,"""M020100""

In [73]:
data.shape

(62783, 61)

In [74]:
data.null_count()

trip_id,start_date,stop_sequence,feed_timestamp,fetch_timestamp,start_time,schedule_relationship,vehicle_id,trip_timestamp,stop_id,arrival_time,departure_time,event_time,route_id,direction_id,shape_id,service_id,stop_id_right,scheduled_arrival,scheduled_departure,stop_lat,stop_lon,vehicle_time,latitude,longitude,bearing,speed_mps,next_arrival_time,travel_time,event_time_local,hour,weekday,month,is_peak,scheduled_segment_time,trip_progress,next_stop_id,segment_length,segment_scheduled_travel_time,scheduled_segment_speed_mps,temperature_c,precipitation_mm,rain_mm,snowfall_cm,windspeed_kmh,weathercode,is_raining,is_snowing,is_fog,service_date,scheduled_arrival_epoch,delay_seconds,ridership,transfers,upstream_delay_seconds,headway_seconds,is_weekend,is_federal_holiday,is_school_day,major_event_count,has_major_event
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3738,3738,3738,3738,3761,0,0,0,0,0,0,0,1219,0,1219,3699,3699,3699,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1219,690,0,0,0,0,0


In [75]:
features = [
    "route_id",
    "direction_id",
    "shape_id",
    "service_id",
 
    "stop_sequence",
    "trip_progress",
 
    "hour",
    "weekday",
    "month",
    "is_peak",
 
    "scheduled_arrival",
    "scheduled_departure",
    "scheduled_segment_time",
 
    "stop_lat",
    "stop_lon",
 
    "latitude",
    "longitude",
    "bearing",
    "temperature_c",
    "precipitation_mm",
    "snowfall_cm",
    "windspeed_kmh",
    "is_raining",
    "is_snowing",
    "is_fog",
    "weathercode",
    "segment_length",
    "scheduled_segment_speed_mps",
    "upstream_delay_seconds",
    "speed_mps",
    "headway_seconds",
    "is_weekend",
    "is_federal_holiday",
    "is_school_day",
    "has_major_event",

    
    "ridership",
    "transfers",
]
target = "travel_time"
id_cols = ["trip_id", "start_date"]

print(data.columns)
 
# %%
model_ready = data.select(id_cols + features + [target]).drop_nulls(subset=features + [target])
 
print(model_ready.shape)
print(model_ready.null_count())
print(model_ready.head())

['trip_id', 'start_date', 'stop_sequence', 'feed_timestamp', 'fetch_timestamp', 'start_time', 'schedule_relationship', 'vehicle_id', 'trip_timestamp', 'stop_id', 'arrival_time', 'departure_time', 'event_time', 'route_id', 'direction_id', 'shape_id', 'service_id', 'stop_id_right', 'scheduled_arrival', 'scheduled_departure', 'stop_lat', 'stop_lon', 'vehicle_time', 'latitude', 'longitude', 'bearing', 'speed_mps', 'next_arrival_time', 'travel_time', 'event_time_local', 'hour', 'weekday', 'month', 'is_peak', 'scheduled_segment_time', 'trip_progress', 'next_stop_id', 'segment_length', 'segment_scheduled_travel_time', 'scheduled_segment_speed_mps', 'temperature_c', 'precipitation_mm', 'rain_mm', 'snowfall_cm', 'windspeed_kmh', 'weathercode', 'is_raining', 'is_snowing', 'is_fog', 'service_date', 'scheduled_arrival_epoch', 'delay_seconds', 'ridership', 'transfers', 'upstream_delay_seconds', 'headway_seconds', 'is_weekend', 'is_federal_holiday', 'is_school_day', 'major_event_count', 'has_major_e

In [76]:
model_ready.columns

['trip_id',
 'start_date',
 'route_id',
 'direction_id',
 'shape_id',
 'service_id',
 'stop_sequence',
 'trip_progress',
 'hour',
 'weekday',
 'month',
 'is_peak',
 'scheduled_arrival',
 'scheduled_departure',
 'scheduled_segment_time',
 'stop_lat',
 'stop_lon',
 'latitude',
 'longitude',
 'bearing',
 'temperature_c',
 'precipitation_mm',
 'snowfall_cm',
 'windspeed_kmh',
 'is_raining',
 'is_snowing',
 'is_fog',
 'weathercode',
 'segment_length',
 'scheduled_segment_speed_mps',
 'upstream_delay_seconds',
 'speed_mps',
 'headway_seconds',
 'is_weekend',
 'is_federal_holiday',
 'is_school_day',
 'has_major_event',
 'ridership',
 'transfers',
 'travel_time']

In [77]:
model_ready

trip_id,start_date,route_id,direction_id,shape_id,service_id,stop_sequence,trip_progress,hour,weekday,month,is_peak,scheduled_arrival,scheduled_departure,scheduled_segment_time,stop_lat,stop_lon,latitude,longitude,bearing,temperature_c,precipitation_mm,snowfall_cm,windspeed_kmh,is_raining,is_snowing,is_fog,weathercode,segment_length,scheduled_segment_speed_mps,upstream_delay_seconds,speed_mps,headway_seconds,is_weekend,is_federal_holiday,is_school_day,has_major_event,ridership,transfers,travel_time
str,str,str,i64,str,str,u32,f64,i8,i8,i8,i8,i32,i32,i32,f64,f64,f32,f32,f32,f64,f64,f64,f64,i8,i8,i8,i64,f64,f64,i64,f64,i64,bool,bool,bool,bool,i64,i64,i64
"""MV_A6-Weekday-SDon-004000_M2_2…","""20260312""","""M2""",0,"""M020100""","""MV_A6-Weekday-SDon""",3,0.272727,0,4,3,0,2467,2467,31,40.82426,-73.936943,40.824047,-73.93721,53.130104,17.3,2.2,0.0,13.9,1,0,0,61,171.64037,5.048246,506,1.785702,97,false,false,true,false,60,3,13
"""MV_A6-Weekday-SDon-004000_M2_2…","""20260312""","""M2""",0,"""M020100""","""MV_A6-Weekday-SDon""",4,0.363636,0,4,3,0,2504,2504,37,40.825612,-73.935955,40.824047,-73.93721,53.130104,17.3,2.2,0.0,13.9,1,0,0,61,118.236767,5.140729,523,1.785702,81,false,false,true,false,60,3,43
"""MV_A6-Weekday-SDon-004000_M2_2…","""20260312""","""M2""",0,"""M020100""","""MV_A6-Weekday-SDon""",5,0.454545,0,4,3,0,2530,2530,26,40.826531,-73.935264,40.827339,-73.935211,97.594643,17.3,2.2,0.0,13.9,1,0,0,61,779.224571,5.027255,499,79.063506,99,false,false,true,false,60,3,171
"""MV_A6-Weekday-SDon-004000_M2_2…","""20260312""","""M2""",0,"""M020100""","""MV_A6-Weekday-SDon""",6,0.545455,0,4,3,0,2702,2702,172,40.830904,-73.940279,40.830944,-73.940338,93.63121,17.3,2.2,0.0,13.9,1,0,0,61,243.057468,5.063697,516,63.665023,87,false,false,true,false,60,3,35
"""MV_A6-Weekday-SDon-004000_M2_2…","""20260312""","""M2""",0,"""M020100""","""MV_A6-Weekday-SDon""",9,0.818182,0,4,3,0,2824,2824,69,40.835242,-73.937496,40.834999,-73.937698,65.148415,17.3,2.2,0.0,13.9,1,0,0,61,166.574166,5.047702,497,11.281706,81,false,false,true,false,60,3,29
"""MV_A6-Weekday-SDon-004000_M2_2…","""20260312""","""M2""",0,"""M020100""","""MV_A6-Weekday-SDon""",10,0.909091,0,4,3,0,2860,2860,36,40.836657,-73.936852,40.834999,-73.937698,65.148415,17.3,2.2,0.0,13.9,1,0,0,61,204.714537,4.993037,448,11.281706,80,false,false,true,false,60,3,44
"""MV_A6-Weekday-SDon-006600_M2_2…","""20260312""","""M2""",1,"""M020110""","""MV_A6-Weekday-SDon""",3,0.041667,1,4,3,0,4017,4017,57,40.839624,-73.940184,40.839294,-73.939911,266.82016,17.4,4.6,0.0,15.9,1,0,0,63,322.407846,4.960121,164,2.883099,3734,false,false,true,false,9,1,93
"""MV_A6-Weekday-SDon-006600_M2_2…","""20260312""","""M2""",1,"""M020110""","""MV_A6-Weekday-SDon""",4,0.055556,1,4,3,0,4075,4075,58,40.837687,-73.93804,40.837799,-73.938187,337.757233,17.4,4.6,0.0,15.9,1,0,0,63,242.833097,5.059023,148,1.853977,3735,false,false,true,false,9,1,85
"""MV_A6-Weekday-SDon-006600_M2_2…","""20260312""","""M2""",1,"""M020110""","""MV_A6-Weekday-SDon""",5,0.069444,1,4,3,0,4119,4119,44,40.836219,-73.937231,40.834759,-73.937805,246.801407,17.4,4.6,0.0,15.9,1,0,0,63,141.837214,4.890938,183,5.451419,3705,false,false,true,false,9,1,5


In [78]:
print(model_ready.describe())

shape: (9, 41)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬────────┐
│ sta ┆ tri ┆ sta ┆ rou ┆ dir ┆ sha ┆ ser ┆ sto ┆ tri ┆ hou ┆ wee ┆ mon ┆ is_ ┆ sch ┆ sch ┆ sch ┆ sto ┆ sto ┆ lat ┆ lon ┆ bea ┆ tem ┆ pre ┆ sno ┆ win ┆ is_ ┆ is_ ┆ is_ ┆ wea ┆ seg ┆ sch ┆ ups ┆ spe ┆ hea ┆ is_ ┆ is_ ┆ is_ ┆ has ┆ rid ┆ tra ┆ travel │
│ tis ┆ p_i ┆ rt_ ┆ te_ ┆ ect ┆ pe_ ┆ vic ┆ p_s ┆ p_p ┆ r   ┆ kda ┆ th  ┆ pea ┆ edu ┆ edu ┆ edu ┆ p_l ┆ p_l ┆ itu ┆ git ┆ rin ┆ per ┆ cip ┆ wfa ┆ dsp ┆ rai ┆ sno ┆ fog ┆ the ┆ men ┆ edu ┆ tre ┆ ed_ ┆ dwa ┆ wee ┆ fed ┆ sch ┆ _ma ┆ ers ┆ nsf ┆ _time  │
│ tic ┆ d   ┆ dat ┆ id  ┆ ion ┆ id  ┆ e_i ┆ equ ┆ rog ┆ --- ┆ y   ┆ --- ┆ k   ┆ led ┆ led ┆ led ┆ at  ┆ on  ┆ de  ┆ ude ┆ g   ┆ atu ┆ ita ┆ ll_ ┆ eed ┆ nin ┆ win ┆ --- ┆ rco ┆ t_l ┆ led ┆ am_ ┆ mps ┆ y_s ┆ ken ┆ era ┆ ool ┆ jor ┆ hi

In [79]:
model_ready.group_by("route_id").len().sort("len", descending=True)

route_id,len
str,u32
"""M4""",13284
"""M101""",12386
"""M15""",11603
"""M1""",9838
"""M2""",7609


In [80]:
model_ready.select([
    pl.col("scheduled_arrival").min().alias("min"),
    pl.col("scheduled_arrival").max().alias("max"),
    pl.col("scheduled_arrival").mean().alias("mean"),
])

min,max,mean
i32,i32,f64
1440,95134,50657.920541


In [81]:
print(
    trip_updates.group_by("route_id").len().sort("len", descending=True)
)

print(
    matched.group_by("route_id").len().sort("len", descending=True)
)

print(
    model_ready.group_by("route_id").len().sort("len", descending=True)
)

shape: (5, 2)
┌──────────┬───────┐
│ route_id ┆ len   │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ M4       ┆ 17256 │
│ M101     ┆ 15820 │
│ M15      ┆ 15022 │
│ M1       ┆ 12767 │
│ M2       ┆ 10401 │
└──────────┴───────┘
shape: (5, 2)
┌──────────┬───────┐
│ route_id ┆ len   │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ M4       ┆ 17256 │
│ M101     ┆ 15820 │
│ M15      ┆ 15022 │
│ M1       ┆ 12767 │
│ M2       ┆ 10401 │
└──────────┴───────┘
shape: (5, 2)
┌──────────┬───────┐
│ route_id ┆ len   │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ M4       ┆ 13284 │
│ M101     ┆ 12386 │
│ M15      ┆ 11603 │
│ M1       ┆ 9838  │
│ M2       ┆ 7609  │
└──────────┴───────┘


In [82]:
model_ready.write_parquet("raw/processed_gtfs/baseline_dataset.parquet")
print("written")

written
